# Explore the synthetic process data

Inspect run counts, sampling and labels, feature distributions, representative fault trajectories, and the baseline and fault-specific causal relationships. Run `0.0-generate_process_dataset.ipynb` and `0.2-data_preprocessing.ipynb` first.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [2]:
import pandas as pd
from IPython.display import display

from carl.causalai_process import VARIABLE_NAMES


SOURCE_DIR = PROJECT_ROOT / "data" / "processed" / "causalAIProcess"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "interim" / "causalAIProcess"
required_paths = [
    SOURCE_DIR / "traces.parquet",
    SOURCE_DIR / "runs.csv",
    SOURCE_DIR / "causal_graph.csv",
    SOURCE_DIR / "fault_coefficients.csv",
    PREPROCESSED_DIR / "traces_scaled.parquet",
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Required generated/preprocessed files are missing: "
        + ", ".join(str(path) for path in missing_paths)
        + ". Run notebooks 0.0 and 0.2 first."
    )

raw_traces = pd.read_parquet(SOURCE_DIR / "traces.parquet")
traces = pd.read_parquet(PREPROCESSED_DIR / "traces_scaled.parquet")
runs = pd.read_csv(SOURCE_DIR / "runs.csv")
causal_graph = pd.read_csv(SOURCE_DIR / "causal_graph.csv")
fault_coefficients = pd.read_csv(SOURCE_DIR / "fault_coefficients.csv")
fault_type_by_run = runs.set_index("run_id")["fault_type"]
raw_traces["fault_type"] = raw_traces["run_id"].map(fault_type_by_run)
traces["fault_type"] = traces["run_id"].map(fault_type_by_run)
fault_types = ["healthy", *runs.loc[runs["fault_type"] != "healthy", "fault_type"].drop_duplicates()]

assert len(raw_traces) == len(traces)
assert raw_traces["run_id"].equals(traces["run_id"])
assert raw_traces["time_step"].equals(traces["time_step"])
assert traces["anomaly"].equals(raw_traces["anomaly"])
assert set(VARIABLE_NAMES).issubset(traces.columns)
print(f"Runs: {runs['run_id'].nunique()}; samples: {len(traces):,}")
display(runs.groupby(["fault_type", "split"]).size().unstack(fill_value=0))
display(runs.groupby("fault_type")["fault_start_step"].agg(["count", "min", "max"]))

Runs: 2000; samples: 2,000,000


split,test,train,validation
fault_type,,,
healthy,75,350,75
hydraulic_coupling_shift,75,350,75
reaction_coupling_shift,75,350,75
thermal_coupling_shift,75,350,75


,count,min,max
fault_type,,,
healthy,500,-1,-1
hydraulic_coupling_shift,500,200,200
reaction_coupling_shift,500,200,200
thermal_coupling_shift,500,200,200


In [3]:
print("Raw process-unit summary by fault type:")
display(raw_traces.groupby("fault_type")[list(VARIABLE_NAMES)].agg(["mean", "std"]).round(3))

print("Robust-scaled features by split and anomaly label:")
display(
    traces.groupby(["split", "anomaly"])[list(VARIABLE_NAMES)]
    .agg(["mean", "std"])
    .round(3)
)

Raw process-unit summary by fault type:


feed_flow         feed_concentration         \
                              mean     std               mean    std   
fault_type                                                             
healthy                     99.989   6.277              1.001  0.058   
hydraulic_coupling_shift   100.022  12.165              1.007  0.119   
reaction_coupling_shift    100.029  12.106              1.009  0.119   
thermal_coupling_shift     100.018   4.991              1.001  0.048   

                         coolant_flow         reactor_concentration         \
                                 mean     std                  mean    std   
fault_type                                                                   
healthy                       100.023   6.269                 0.700  0.021   
hydraulic_coupling_shift      100.032  12.161                 0.701  0.027   
reaction_coupling_shift        99.799  12.058                 0.722  0.144   
thermal_coupling_shift        100.009   5.008                 0.700  0.046   

                         reactor_temperature         product_quality         
                                        mean     std            mean    std  
fault_type                                                                   
healthy                              349.987   4.729           0.850  0.002  
hydraulic_coupling_shift             349.995   5.589           0.850  0.002  
reaction_coupling_shift              353.114  59.554           0.849  0.009  
thermal_coupling_shift               349.164  31.746           0.850  0.007

Robust-scaled features by split and anomaly label:


feed_flow        feed_concentration        coolant_flow  \
                        mean    std               mean    std         mean   
split      anomaly                                                           
test       0          -0.007  0.750              0.015  0.746        0.004   
           1          -0.015  1.345              0.082  1.456       -0.024   
train      0           0.001  0.756              0.024  0.758       -0.001   
           1           0.003  1.336              0.094  1.410       -0.008   
validation 0          -0.004  0.762              0.025  0.757        0.004   
           1           0.006  1.339              0.091  1.423       -0.015   

                          reactor_concentration        reactor_temperature  \
                      std                  mean    std                mean   
split      anomaly                                                           
test       0        0.754                 0.011  0.742              -0.009   
           1        1.357                 0.212  3.640              -0.140   
train      0        0.755                 0.016  0.749               0.001   
           1        1.333                 0.390  3.551               0.208   
validation 0        0.757                 0.020  0.749               0.002   
           1        1.312                 0.304  3.451               0.188   

                          product_quality         
                      std            mean    std  
split      anomaly                                
test       0        0.743          -0.006  0.744  
           1        7.274           0.074  2.440  
train      0        0.747          -0.001  0.752  
           1        6.908          -0.166  2.423  
validation 0        0.750          -0.004  0.752  
           1        6.521          -0.058  2.335

In [4]:
representative_runs = {
    fault_type: runs.loc[runs["fault_type"] == fault_type].iloc[0]
    for fault_type in fault_types
}
window_summaries = []
sampled_trajectories = []
for fault_type, run in representative_runs.items():
    trajectory = traces.loc[traces["run_id"] == run["run_id"]].sort_values("time_step")
    onset = int(run["fault_start_step"])
    windows = {
        "before_fault": trajectory.loc[trajectory["time_step"] < onset]
        if onset >= 0
        else trajectory,
        "after_fault": trajectory.loc[trajectory["time_step"] >= onset]
        if onset >= 0
        else trajectory.iloc[0:0],
    }
    for window_name, window in windows.items():
        if window.empty:
            continue
        for variable in VARIABLE_NAMES:
            window_summaries.append(
                {
                    "fault_type": fault_type,
                    "window": window_name,
                    "variable": variable,
                    "mean_scaled": window[variable].mean(),
                    "std_scaled": window[variable].std(),
                }
            )
    sample_steps = set(
        range(0, len(trajectory), max(1, len(trajectory) // 12))
    )
    if onset >= 0:
        sample_steps.add(onset)
    sampled = trajectory.loc[trajectory["time_step"].isin(sample_steps)]
    sampled_trajectories.append(
        sampled[["run_id", "time_step", "anomaly", *VARIABLE_NAMES]].assign(
            fault_type=fault_type
        )
    )

print("Representative-run scaled feature summaries before and after fault onset:")
display(pd.DataFrame(window_summaries).set_index(["fault_type", "window", "variable"]).round(3))
print("Downsampled representative trajectories (the exact onset sample is included):")
display(pd.concat(sampled_trajectories, ignore_index=True))

Representative-run scaled feature summaries before and after fault onset:


mean_scaled  \
fault_type               window       variable                             
healthy                  before_fault feed_flow                    0.042   
                                      feed_concentration          -0.034   
                                      coolant_flow                -0.056   
                                      reactor_concentration       -0.086   
                                      reactor_temperature         -0.007   
                                      product_quality             -0.125   
hydraulic_coupling_shift before_fault feed_flow                    0.187   
                                      feed_concentration          -0.106   
                                      coolant_flow                -0.266   
                                      reactor_concentration       -0.117   
                                      reactor_temperature          0.089   
                                      product_quality             -0.072   
                         after_fault  feed_flow                    0.444   
                                      feed_concentration          -1.091   
                                      coolant_flow                 0.234   
                                      reactor_concentration       -0.666   
                                      reactor_temperature         -0.107   
                                      product_quality             -0.150   
thermal_coupling_shift   before_fault feed_flow                   -0.143   
                                      feed_concentration           0.041   
                                      coolant_flow                 0.122   
                                      reactor_concentration        0.148   
                                      reactor_temperature          0.013   
                                      product_quality             -0.027   
                         after_fault  feed_flow                   -0.023   
                                      feed_concentration           0.024   
                                      coolant_flow                 0.011   
                                      reactor_concentration        0.062   
                                      reactor_temperature         -0.187   
                                      product_quality              0.270   
reaction_coupling_shift  before_fault feed_flow                    0.047   
                                      feed_concentration          -0.070   
                                      coolant_flow                 0.035   
                                      reactor_concentration       -0.191   
                                      reactor_temperature         -0.135   
                                      product_quality             -0.029   
                         after_fault  feed_flow                    0.074   
                                      feed_concentration           0.159   
                                      coolant_flow                -0.132   
                                      reactor_concentration        0.271   
                                      reactor_temperature          0.296   
                                      product_quality             -0.183   

                                                             std_scaled  
fault_type               window       variable                           
healthy                  before_fault feed_flow                   0.693  
                                      feed_concentration          0.701  
                                      coolant_flow                0.708  
                                      reactor_concentration       0.664  
                                      reactor_temperature         0.719  
                                      product_quality             0.722  
hydraulic_coupling_shift before_fault feed_flow                   0.819  
                                      feed_concent

Downsampled representative trajectories (the exact onset sample is included):


,run_id,time_step,anomaly,feed_flow,feed_concentration,coolant_flow,reactor_concentration,reactor_temperature,product_quality,fault_type
0,run_healthy_00000,0,0,-0.079193,-0.126232,-0.380075,-0.534666,0.534991,-1.617964,healthy
1,run_healthy_00000,83,0,0.127807,0.492781,-1.017549,-0.181639,0.824217,-0.725479,healthy
2,run_healthy_00000,166,0,1.134863,0.530188,-0.602415,-0.896001,-1.398779,-0.764823,healthy
3,run_healthy_00000,249,0,-0.195188,1.010062,-1.391145,1.245404,0.381897,0.384595,healthy
4,run_healthy_00000,332,0,0.175775,-0.367851,-1.298537,-0.556145,-0.301871,-0.972447,healthy
5,run_healthy_00000,415,0,0.274248,1.627875,-0.422018,0.096163,0.371891,-0.100215,healthy
6,run_healthy_00000,498,0,-0.892856,0.246181,-0.067452,-0.374369,0.407164,-0.202532,healthy
7,run_healthy_00000,581,0,-0.567494,-0.517662,-0.294559,0.119492,0.537815,0.068635,healthy
8,run_healthy_00000,664,0,0.345486,0.165815,-0.312576,0.218640,1.423495,-1.719490,healthy
9,run_healthy_00000,747,0,0.350503,0.530981,0.355382,0.242675,1.111048,-0.168708,healthy


## Causal structure and per-instance changes

The graph topology is shared across runs. Fault type controls the direction of each edge's coefficient change, while randomized change magnitudes make coefficient values instance-specific. The tables below show the baseline graph and the changed coefficients for one run of each fault type.

In [5]:
display(causal_graph.sort_values(["target", "source", "lag_steps"]).reset_index(drop=True))
for fault_type in fault_types[1:]:
    run_id = runs.loc[runs["fault_type"] == fault_type, "run_id"].iloc[0]
    print(f"{fault_type} — {run_id}")
    display(
        fault_coefficients.loc[fault_coefficients["run_id"] == run_id]
        .sort_values(["target", "source", "lag_steps"])
        .reset_index(drop=True)
    )

,source,target,lag_steps,coefficient,function,mechanism
0,coolant_flow,coolant_flow,1,0.75,identity,input_persistence
1,feed_concentration,feed_concentration,1,0.70,identity,input_persistence
2,feed_flow,feed_flow,1,0.75,identity,input_persistence
3,product_quality,product_quality,1,0.55,identity,state_persistence
4,reactor_concentration,product_quality,1,0.22,tanh,conversion
5,reactor_temperature,product_quality,2,-0.16,tanh,temperature_sensitivity
6,feed_concentration,reactor_concentration,1,0.22,tanh,feed_mixing
7,feed_flow,reactor_concentration,1,-0.12,tanh,hydraulic_residence
8,reactor_concentration,reactor_concentration,1,0.62,identity,state_persistence
9,reactor_temperature,reactor_concentration,1,0.08,tanh,reaction_kinetics


hydraulic_coupling_shift — run_hydraulic_coupling_shift_00500


,run_id,source,target,lag_steps,coefficient_before,coefficient_after,change_fraction,requested_change_fraction
0,run_hydraulic_coupling_shift_00500,coolant_flow,coolant_flow,1,0.75,0.950000,0.266667,0.494356
1,run_hydraulic_coupling_shift_00500,feed_concentration,feed_concentration,1,0.70,0.950000,0.357143,0.378620
2,run_hydraulic_coupling_shift_00500,feed_flow,feed_flow,1,0.75,0.950000,0.266667,0.451788
3,run_hydraulic_coupling_shift_00500,product_quality,product_quality,1,0.55,0.339130,0.383401,0.383401
4,run_hydraulic_coupling_shift_00500,reactor_concentration,product_quality,1,0.22,0.129630,0.410773,0.410773
5,run_hydraulic_coupling_shift_00500,reactor_temperature,product_quality,2,-0.16,-0.220374,0.377335,0.377335
6,run_hydraulic_coupling_shift_00500,feed_concentration,reactor_concentration,1,0.22,0.311146,0.414299,0.414299
7,run_hydraulic_coupling_shift_00500,feed_flow,reactor_concentration,1,-0.12,-0.063968,0.466930,0.466930
8,run_hydraulic_coupling_shift_00500,reactor_concentration,reactor_concentration,1,0.62,0.340067,0.451505,0.451505
9,run_hydraulic_coupling_shift_00500,reactor_temperature,reactor_concentration,1,0.08,0.113788,0.422349,0.422349


thermal_coupling_shift — run_thermal_coupling_shift_01000


,run_id,source,target,lag_steps,coefficient_before,coefficient_after,change_fraction,requested_change_fraction
0,run_thermal_coupling_shift_01000,coolant_flow,coolant_flow,1,0.75,0.387390,0.483481,0.483481
1,run_thermal_coupling_shift_01000,feed_concentration,feed_concentration,1,0.70,0.382578,0.453460,0.453460
2,run_thermal_coupling_shift_01000,feed_flow,feed_flow,1,0.75,0.430828,0.425563,0.425563
3,run_thermal_coupling_shift_01000,product_quality,product_quality,1,0.55,0.803555,0.461009,0.461009
4,run_thermal_coupling_shift_01000,reactor_concentration,product_quality,1,0.22,0.305020,0.386453,0.386453
5,run_thermal_coupling_shift_01000,reactor_temperature,product_quality,2,-0.16,-0.093853,0.413418,0.413418
6,run_thermal_coupling_shift_01000,feed_concentration,reactor_concentration,1,0.22,0.120403,0.452714,0.452714
7,run_thermal_coupling_shift_01000,feed_flow,reactor_concentration,1,-0.12,-0.169750,0.414583,0.414583
8,run_thermal_coupling_shift_01000,reactor_concentration,reactor_concentration,1,0.62,0.859308,0.385980,0.385980
9,run_thermal_coupling_shift_01000,reactor_temperature,reactor_concentration,1,0.08,0.044321,0.445982,0.445982


reaction_coupling_shift — run_reaction_coupling_shift_01500


,run_id,source,target,lag_steps,coefficient_before,coefficient_after,change_fraction,requested_change_fraction
0,run_reaction_coupling_shift_01500,coolant_flow,coolant_flow,1,0.75,0.949477,0.265969,0.265969
1,run_reaction_coupling_shift_01500,feed_concentration,feed_concentration,1,0.70,0.912055,0.302936,0.302936
2,run_reaction_coupling_shift_01500,feed_flow,feed_flow,1,0.75,0.950000,0.266667,0.303871
3,run_reaction_coupling_shift_01500,product_quality,product_quality,1,0.55,0.702500,0.277273,0.277273
4,run_reaction_coupling_shift_01500,reactor_concentration,product_quality,1,0.22,0.154209,0.299051,0.299051
5,run_reaction_coupling_shift_01500,reactor_temperature,product_quality,2,-0.16,-0.203514,0.271965,0.271965
6,run_reaction_coupling_shift_01500,feed_concentration,reactor_concentration,1,0.22,0.160912,0.268580,0.268580
7,run_reaction_coupling_shift_01500,feed_flow,reactor_concentration,1,-0.12,-0.082760,0.310329,0.310329
8,run_reaction_coupling_shift_01500,reactor_concentration,reactor_concentration,1,0.62,0.796197,0.284189,0.284189
9,run_reaction_coupling_shift_01500,reactor_temperature,reactor_concentration,1,0.08,0.105281,0.316007,0.316007
